In [ ]:
%run main.ipynb

   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
3  5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
4  3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   

   Longitude  
0    -122.23  
1    -122.22  
2    -122.24  
3    -122.25  
4    -122.25  
0    4.526
1    3.585
2    3.521
3    3.413
4    3.422
Name: MedHouseVal, dtype: float64
Shape of X: (20640, 8)
Missing values before cleaning:
MedInc        0
HouseAge      0
AveRooms      0
AveBedrms     0
Population    0
AveOccup      0
Latitude      0
Longitude     0
dtype: int64
Shape after removing missing values: (20640, 8)
Architecture: (32,), Activation: tanh, Validation MSE: 0.3137
Architecture: (32,), Activation: relu, Validation MSE: 0.3

/opt/anaconda3/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:697: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")


Architecture: (64, 32), Activation: tanh, Validation MSE: 0.2888
Architecture: (64, 32), Activation: relu, Validation MSE: 0.2890


/opt/anaconda3/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:697: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")


Architecture: (64, 32), Activation: logistic, Validation MSE: 0.7062


In [ ]:
#Baseline Modal
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error

# 先分出 test
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=26
)
#X_train_val → 80% 的特征，X_test → 20% 的特征，y_train_val → 80% 的标签，y_test → 20% 的标签

# 从X_train_val, y_train_val里再分 validation（分成X_train, X_val, y_train, y_val）
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.1, random_state=26
)#这里的0.1是80%里的10%


# 在标准化前，先保存 raw（未标准化）版本：后面kNN需要用原始经纬度
X_train_raw = X_train.copy()
X_val_raw   = X_val.copy()
X_test_raw  = X_test.copy()

# scale (fit only on train)#标准化（只在训练集）并转回DataFrame
scaler = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns,index=X_train_raw.index)
X_val   = pd.DataFrame(scaler.transform(X_val), columns=X.columns,index=X_val_raw.index)
X_test  = pd.DataFrame(scaler.transform(X_test), columns=X.columns, index=X_test_raw.index)
#scaler.transform 标准化 test 数据
#index=X_test_raw.index Pandas 会默认生成新的 0~n 索引
#加了 index=...，这样索引不会乱（对齐 y / 后面 debug 更稳）

# model (no internal early stopping, since you already have X_val)
model = MLPRegressor(
    hidden_layer_sizes=(64, 32),
    activation='tanh',
    solver='adam',
    learning_rate_init=1e-3,
    max_iter=300,
    early_stopping=False,
    random_state=26
)

model.fit(X_train, y_train)
#测试集和预测集的预测
val_pred = model.predict(X_val)
test_pred = model.predict(X_test)

# 训练集预测
train_pred = model.predict(X_train)
# 计算训练误差
train_mse = mean_squared_error(y_train, train_pred)

print("Baseline Train MSE:", train_mse)#加入是为了判断是不是欠拟合或者过拟合
print("Validation MSE:", mean_squared_error(y_val, val_pred))#Mean Squared Error（均方误差）
print("Test MSE:", mean_squared_error(y_test, test_pred))

In [ ]:
#检测训练集中是不是有出现多个样本经纬度完全一样
print("numbers of Duplicate coordinates：",X_train_raw[['Latitude','Longitude']].duplicated().sum())

duplicates = X_train_raw[X_train_raw[['Latitude','Longitude']].duplicated(keep=False)]

print(duplicates[['Latitude','Longitude']].sort_values(['Latitude','Longitude']))

X_train_raw.groupby(['Latitude','Longitude']).size().sort_values(ascending=False).head()
#训练集中确实有大量“经纬度完全相同”的样本
#对某个点来说，不止一个距离=0 的邻居，这会让 idxs = idxs[1:] 这种“删第一个当作自己”的逻辑不够严谨。
#只删掉第一个 idxs[1:]：可能删掉了“同坐标的另一个样本”,但仍然可能保留了自己的那条样本（如果它不是排在第一个）
#也可能保留多个 distance=0 的点（包含自己的 y），有轻微标签泄漏风险
#不能使用下列代码
'''
ArithmeticErrordef neighbour_mean_price(query_df, y_train_series, exclude_self=False):
    """
    为 query_df 的每个样本计算：训练集中k个邻居的 y 的平均值
    query_df: 需要计算邻居特征的数据（train/val/test 的 raw DataFrame）
    y_train_series: 训练集标签（只能用它，避免data leakage）
    exclude_self: 如果 query_df 是训练集本身，要剔除“自己”这个邻居
    """
    # indices：每个query点在训练集中对应的k个邻居的索引
    _, indices = knn.kneighbors(query_df[['Latitude', 'Longitude']])

    neigh_mean = []  # 存放每个样本的邻居均价
    for idxs in indices:
        if exclude_self:
            # train 查询时，第一个邻居通常是自己，要去掉
            idxs = idxs[1:K+1]
        else:
            # val/test 直接取前K个邻居（它们全来自train）
            idxs = idxs[:K]
        # 用训练集的 y_train 取邻居价格并求平均（关键：不使用val/test真实y）
        neigh_mean.append(y_train_series.iloc[idxs].mean())

    return np.array(neigh_mean)
'''

# 1. 统计每个坐标出现多少次
coord_counts = X_train_raw.groupby(['Latitude', 'Longitude']).size()

# 2. 统计“出现次数”的分布
distribution = coord_counts.value_counts().sort_index()

# 3. 转成表格并加列名
distribution_df = distribution.reset_index()
distribution_df.columns = ['Repetition frequency of coordinate points', 'number of coordinate points']

# 4. 打印标题和表格
print("\n=== Repetitive statistics based on latitude and longitude ===")
print(distribution_df)

#用knn的局限性
#因为有非常多的坐标重复，所以会导致加入邻居特征时，在删除自己之后选择k个邻居选择到的领居出现不一样的情况。
#这种情况发生在和删除掉原坐标完全相同的坐标的情况非常少，在K值选择7的时候，有3个坐标，32个重复样本
#sklearn 在 多个距离相同的邻居之间没有唯一排序，返回的邻居集合可能因为内部实现顺序不同而变化
#用random seeds不能避免这种情况，random_state 只控制“随机过程”，只影响train_test_split，MLPRegressor 的权重初始化和可能的内部随机优化过程
#这种影响预测率的情况不是一定会发生，只是可能存在这种风险
#虽然代码每次运行其实返回的是同一组邻居（因为数据顺序不变，sklearn版本不变，算法参数不变），但sklearn实际上在距离相同的情况下就按内部顺序返回前 K 个
#有可能内部顺序改变一下可能预测准确率会提高或下降，所以我们找到的可能不是最优的参数解

'''
| 方法                 | 问题                |
| ------------------ | ----------------- |
| kNN                | 邻居数量固定，但邻居组合可能不唯一 |
| distance threshold | 邻居组合唯一，但邻居数量不固定   |
'''

In [ ]:
# ============================================================
# 第二个模型：加入 neighbourhood feature（NeighbourPrice）
# ============================================================

from sklearn.neighbors import NearestNeighbors  # 用于找k个最近邻

K = 7  # 邻居数量k（你们分支实验可改成3/5/10，但一次实验只改k）找到距离它最近的 7 套房子

# ------------------------------------------------------------
# 1) 用训练集的经纬度建立 kNN（只fit训练集）
#    这样 val/test 找邻居时，也只会在 train 里找（避免用到test标签）
# ------------------------------------------------------------
knn = NearestNeighbors(n_neighbors=K+1)  # 创建kNN搜索器
knn.fit(X_train_raw[['Latitude', 'Longitude']])  # 只用训练集坐标来fit

def neighbour_mean_price(query_df, y_train_series, exclude_self=False):
    """
    为 query_df 的每个样本计算：训练集中k个邻居的 y 的平均值
    query_df: 需要计算邻居特征的数据（train/val/test 的 raw DataFrame）
    y_train_series: 训练集标签（只能用它，避免data leakage）
    exclude_self: 如果 query_df 是训练集本身，要剔除“自己”这个邻居
    """

    #这行代码的作用是把 y_train_series 的顺序重新排列，使它与 X_train_raw 的索引顺序完全一致，从而保证每一行特征对应正确的标签。
    y_train_series = y_train_series.loc[X_train_raw.index]
    #根据每个样本的经纬度，在训练集中找到最近的 K 个邻居，并返回这些邻居的距离 (dists) 和在训练集中的位置索引 (inds)
    dists, inds = knn.kneighbors(query_df[['Latitude', 'Longitude']])

    neigh_mean = [] ## 存放每个样本的邻居均价
    for row_i, idxs in enumerate(inds):
        if exclude_self:
            # 只剔除“自己这一行”的训练索引（最严谨），剔除自己（完全一样的那行数据）
            idxs = idxs[idxs != row_i]  # 用位置索引剔除自己
            idxs = idxs[:K]                 # 保证还是K个邻居，比如上一行代码要执行删除时，原数据并没有自己defensive programming（防御式编程）
            #比如以后用距离过滤，或者过滤多个点，不会变成 6 个或其他奇怪数量
        else:
            # val/test 直接取前K个邻居（它们全来自train）
            idxs = idxs[:K]
        # 用训练集的 y_train 取邻居价格并求平均（关键：不使用val/test真实y）
        neigh_mean.append(y_train_series.iloc[idxs].mean())

    return np.array(neigh_mean)
'''
觉得“我在训练集里拿某一条样本当 query，那 kNN 返回的邻居里怎么可能没有它自己？”
按数学定义，它应该一定在。
但你看到的 Case A（极少数）说明：在实现细节上，它“可能不在返回的前 K+1 个里”。
原因主要是 并列距离（tie）+ 只取前 K+1 个 + 内部取子集。

你并不是“先拿这一条样本再算平均”

你做的是：
用这条样本的经纬度作为 query
让 kneighbors() 在训练集里找 最接近的 n_neighbors 个点
返回这些点的索引
你再用这些索引去取 y_train 求平均

也就是说：
你并不是在邻居列表里手动把“自己”塞进去。
你完全依赖 kneighbors() 返回的那一组邻居
'''
# ------------------------------------------------------------
# 2) 计算 NeighbourPrice（邻居均价）
#    - train：剔除自己
#    - val/test：邻居都来自train，不存在“自己”，不用剔除
# ------------------------------------------------------------
neigh_price_train = neighbour_mean_price(X_train_raw, y_train, exclude_self=True)
neigh_price_val   = neighbour_mean_price(X_val_raw,   y_train, exclude_self=False)
neigh_price_test  = neighbour_mean_price(X_test_raw,  y_train, exclude_self=False)
#exclude_self=True 每个样本最近的点一定是自己，所以删除自己
#验证集样本根本不在训练集中，所以后面不需要删除自己

'''
| 变量          | 作用              |
| ----------- | --------------- |
| `X_val_raw` | 查询点（我想知道它附近房价）  |
| `y_train`   | 数据库（我用它来查邻居的价格） |
'''

# ------------------------------------------------------------
# 3) 把新特征train,val和test 的 neighbour price 加到 raw 数据里，做成新列（注意：从 raw 开始加）
# ------------------------------------------------------------
X_train_enh_raw = X_train_raw.copy()
X_val_enh_raw   = X_val_raw.copy()
X_test_enh_raw  = X_test_raw.copy()

X_train_enh_raw['NeighbourPrice'] = neigh_price_train
X_val_enh_raw['NeighbourPrice']   = neigh_price_val
X_test_enh_raw['NeighbourPrice']  = neigh_price_test

# ------------------------------------------------------------
# 4) 新增列后必须重新标准化（仍然只fit train）
# ------------------------------------------------------------
scaler2 = StandardScaler()
X_train_enh = pd.DataFrame(scaler2.fit_transform(X_train_enh_raw), columns=X_train_enh_raw.columns,index=X_train_enh_raw.index)
X_val_enh   = pd.DataFrame(scaler2.transform(X_val_enh_raw), columns=X_val_enh_raw.columns,index=X_val_enh_raw.index)
X_test_enh  = pd.DataFrame(scaler2.transform(X_test_enh_raw), columns=X_test_enh_raw.columns,index=X_test_enh_raw.index)

# ------------------------------------------------------------
# 5) 训练第二个模型（参数保持与baseline一致，保证公平对比）
# ------------------------------------------------------------
model_enh = MLPRegressor(
    hidden_layer_sizes=(128, 32),      # 与baseline相同
    activation='tanh',
    solver='adam',
    learning_rate_init=1e-3,
    max_iter=800,
    early_stopping=False,
    random_state=26
)

model_enh.fit(X_train_enh, y_train)

# ------------------------------------------------------------
# 6) 评估第二个模型
# ------------------------------------------------------------
val_pred_enh  = model_enh.predict(X_val_enh)
test_pred_enh = model_enh.predict(X_test_enh)

enh_val_mse  = mean_squared_error(y_val, val_pred_enh)
enh_test_mse = mean_squared_error(y_test, test_pred_enh)

train_pred_enh = model_enh.predict(X_train_enh)
enh_train_mse = mean_squared_error(y_train, train_pred_enh)

print("Enhanced Train MSE:", enh_train_mse)
print("Enhanced Validation MSE:", enh_val_mse)#模型在验证集上的平均平方误差是 0.222
print("Enhanced Test MSE:", enh_test_mse)#模型在最终测试集上的平均平方误差是 0.1835

# ------------------------------------------------------------
# 7) 和 baseline 对比（负数表示Enhanced更好）
# ------------------------------------------------------------
print("Delta Test MSE (Enhanced - Baseline):", enh_test_mse - mean_squared_error(y_test, test_pred))
#报告思路是我探索不同领域的参数a=，b=，c=

In [ ]:
# 检查训练集中每个点的第一个邻居是否就是自己（通常为True）
d, ind = knn.kneighbors(X_train_raw[['Latitude','Longitude']])
print("Self in first neighbor:", np.mean(ind[:,0] == np.arange(len(X_train_raw))))

In [ ]:
#判断 训练集每个样本的 kneighbors 返回结果里到底有没有包含自己
import numpy as np

# 用你当前的 knn（已 fit 在 X_train_raw 上），以及当前 K
# 注意：这里检查的是训练集自身 query
dists, inds = knn.kneighbors(X_train_raw[['Latitude', 'Longitude']])

n = len(X_train_raw)

# 对训练集来说，“自己”的训练位置索引就是 0..n-1
self_pos = np.arange(n)

# A: 自己不在邻居列表里
self_in_list = (inds == self_pos[:, None]).any(axis=1)
count_A = np.sum(~self_in_list)

# B: 自己在邻居列表里，但不一定第一个
count_B = np.sum(self_in_list)

# B1: 自己在列表里且是第一个邻居
count_B_first = np.sum(inds[:, 0] == self_pos)

# B2: 自己在列表里但不是第一个
count_B_not_first = np.sum(self_in_list & (inds[:, 0] != self_pos))

print(f"Total train samples: {n}")
print(f"Case A (self NOT in returned neighbors): {count_A} ({count_A/n:.3%})")
print(f"Case B (self IN returned neighbors):     {count_B} ({count_B/n:.3%})")
print(f"  - B1: self is the 1st neighbor:        {count_B_first} ({count_B_first/n:.3%})")
print(f"  - B2: self NOT the 1st neighbor:       {count_B_not_first} ({count_B_not_first/n:.3%})")

# 给几个例子看看（最多打印5个）
if count_A > 0:
    ex = np.where(~self_in_list)[0][:5]
    print("\nExamples of Case A (row_i where self missing):", ex.tolist())
    print("Their neighbor indices (first row shown):")
    print(inds[ex[0]])
    print("Distances of that row:")
    print(dists[ex[0]])

if count_B_not_first > 0:
    ex = np.where(self_in_list & (inds[:, 0] != self_pos))[0][:5]
    print("\nExamples of Case B2 (self present but not first):", ex.tolist())
    r = ex[0]
    pos = np.where(inds[r] == r)[0][0]
    print(f"For row {r}, self appears at position {pos} in neighbor list.")
    print("Neighbor indices:", inds[r])
    print("Distances:", dists[r])

In [ ]:
'''ArithmeticError为什么有early stopping自己还需要划分数据？
因为虽然early stopping划分了数据但early stopping 的 validation 在每一轮模型训练上都会被查看，
因为 early stopping 的 validation 数据参与了训练过程。
如果不提前划分就会产生data leakage的问题。early stopping的作用是判断什么时候停止训练。
'''


main的第三个代码框的输出
Architecture: (32,), Activation: tanh, Validation MSE: 0.3137
Architecture: (32,), Activation: relu, Validation MSE: 0.3262
Architecture: (32,), Activation: logistic, Validation MSE: 0.3210
Architecture: (64,), Activation: tanh, Validation MSE: 0.3140
Architecture: (64,), Activation: relu, Validation MSE: 0.3272
Architecture: (64,), Activation: logistic, Validation MSE: 0.3120
Architecture: (64, 32), Activation: tanh, Validation MSE: 0.2841
Architecture: (64, 32), Activation: relu, Validation MSE: 0.2890
Architecture: (64, 32), Activation: logistic, Validation MSE: 0.3078
Architecture: (128, 32), Activation: tanh, Validation MSE: 0.2789
Architecture: (128, 32), Activation: relu, Validation MSE: 0.2809
Architecture: (128, 32), Activation: logistic, Validation MSE: 0.2899
Architecture: (128, 64), Activation: tanh, Validation MSE: 0.2798
Architecture: (128, 64), Activation: relu, Validation MSE: 0.2793
Architecture: (128, 64), Activation: logistic, Validation MSE: 0.2855
Architecture: (128, 64, 32), Activation: tanh, Validation MSE: 0.3138
Architecture: (128, 64, 32), Activation: relu, Validation MSE: 0.2987
Architecture: (128, 64, 32), Activation: logistic, Validation MSE: 0.2941

=== Validation MSE Table ===
Activation     logistic      relu      tanh
Architecture                               
(32,)          0.320968  0.326200  0.313691
(64,)          0.312038  0.327152  0.313980
(64, 32)       0.307780  0.289044  0.284061
(128, 32)      0.289895  0.280940  0.278872
(128, 64)      0.285536  0.279279  0.279801
(128, 64, 32)  0.294129  0.298666  0.313816

0
0

40
52
2min50s
17min
0.1s
56min20s

第一个模型72/8/20的时候训练的mse
Baseline Train MSE: 0.22346115225375018
Validation MSE: 0.2788722368528284
Test MSE: 0.2503595555373939

变成80/20时候的mse
Baseline Train MSE: 0.22930382604717545
Test MSE: 0.24771853612631176

Project Overview: California Housing Price Prediction using Neural Networks

本项目的目标是使用 California Housing 数据集构建一个房价的神经网络预测模型。
项目首先建立一个 Baseline 神经网络模型，仅使用原始数据特征进行预测。
随后通过构建 neighbourhood-based features（邻域特征），利用地理邻近关系kNN
提取额外信息，从而构建 Enhanced Model，提升了预测性能。

主要流程包括：

1. 数据导入与预处理
   - 加载 California Housing 数据集
   - 处理缺失值
   - 数据标准化
   - 划分训练集、验证集和测试集

2. Baseline Neural Network
   - 构建前馈神经网络 (Feedforward Neural Network)
   - 尝试不同隐藏层结构与激活函数
   - 通过Validation选择最优模型参数
   - 合并Train & Validation 数据集选择最优隐藏层结构与激活函数训练baseline_final_model模型

3. Enhanced Model (Neighbourhood Features)
   - 使用 k-nearest neighbours (kNN) 构建邻域关系
   - 从邻居样本中提取邻居票价的统计特征
   - 研究不同数量的邻居特征对Val MSE的影响
   - 研究相同邻居数量特征与不同k值数量对Val MSE的影响
   - 将最优邻域特征加入原始特征重新训练模型
   - 合并Train & Validation 数据集选择最优邻居特征与k值训练model_enh_final模型

4. baseline_final_model与model_enh_final模型比较
   - 检测最终 Baseline 与 Enhanced 模型是否出现欠拟合或过拟合
   - 比较最终 Baseline 与 Enhanced 模型 MSE 的 improvement percentages
   - 研究最终 Baseline 与 Enhanced 模型在不同房价中预测准确性的影响

5. California Housing 数据集分析（imitation代码部分）
   - 探究数据集中房价高低与出现次数的关系
   - 探索数据集中相同经纬度重复出现的情况
   - 探索数据集中各地区数据的密度情况

5. 代码中的结果可视化
     绘制 Validation MSE for Different Architectures and Activations
   - 绘制 Validation MSE for Feature Combinations
   - 绘制 Validation MSE vs Number of Neighbours (Single Neighbourhood Features)
   - 绘制 Approximate Learning Curves — Overfitting / Underfitting Check\n
   - 绘制 Absolute Error vs Actual Price


PART FIVE

Analysis of the California Housing Dataset (limitation Code Section)
   - Investigate the relationship between house prices and their frequency of occurrence in the dataset
   - Examine instances where the same latitude and longitude coordinates appear multiple times in the dataset
   - Examine the distribution density of data across different regions in the dataset

## Investigate the relationship between house prices and their frequency of occurrence in the dataset

By examining the relationship between the magnitude of house prices and their frequency of occurrence within the dataset, we can determine whether unusually high or low prices might have a disproportionate impact on the MSE, thereby affecting the interpretation of the model’s performance.

In [ ]:
#Explore the relationship between the frequency of occurrence and the high or low prices of houses in the dataset.
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Print the statistical information of the basic price.
print(y.describe())

# 2. Histogram
plt.figure(figsize=(8, 5))
sns.histplot(y, bins=50, kde=True)
plt.title("Distribution of Median House Value")
plt.xlabel("MedHouseVal")
plt.ylabel("Frequency")
plt.show()

# 3. Boxplot
plt.figure(figsize=(8, 3))
sns.boxplot(x=y)
plt.title("Boxplot of Median House Value")
plt.xlabel("MedHouseVal")
plt.show()

# 4. Top 10 highest values
print("Top 10 highest house values:")
print(y.sort_values(ascending=False).head(10))

The house price distribution plot shows that both the magnitude and frequency of house prices generally follow a normal distribution, with the majority of prices concentrated in the lower to middle ranges, whilst a distinct peak in frequency is observed near the maximum value of 5. This indicates that the house price variable in the dataset exhibits a capped value phenomenon, whereby prices exceeding a certain threshold are uniformly recorded as the maximum value. However, the loss function used in this model evaluation relies solely on the mean squared error (MSE). As MSE is sensitive to large errors and outliers, abnormally high property prices and associated errors may exert a disproportionate influence on the MSE, thereby affecting the interpretation of the model’s performance.

## Examine instances where the same latitude and longitude coordinates appear multiple times in the dataset

Determine whether the KNN method suffers from instability in neighbour selection by examining whether there are duplicate latitude and longitude coordinates in the data.

In [ ]:
#Explore the situation where the same latitude and longitude appear repeatedly in the dataset.
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 组合经纬度
# Combine Latitude and Longitude
coords = X[['Longitude', 'Latitude']].copy()

# 统计每个坐标出现次数
#Count the occurrence of each coordinate.
coord_counts = (
    coords.groupby(['Longitude', 'Latitude'])
    .size()
    .reset_index(name='count')
)

# 只保留重复坐标（count > 1）
# Only retain the coordinates that are repeated (where count is greater than 1)
duplicate_coords = coord_counts[coord_counts['count'] > 1].copy()

print("Number of unique coordinates:", len(coord_counts))
print("Number of duplicated coordinate locations:", len(duplicate_coords))
print("Top duplicated coordinates:")
print(duplicate_coords.sort_values('count', ascending=False).head(10))

plt.figure(figsize=(8, 5))
sns.histplot(duplicate_coords['count'], bins=30, edgecolor='black')
plt.title('Distribution of Duplicate Coordinate Counts')
plt.xlabel('Number of samples sharing the same coordinates')
plt.ylabel('Frequency')
plt.show()

This figure reveals that a large number of samples in the dataset share exactly the same geographical coordinates. Of the 20,640 samples, 4,353 coordinate locations contain multiple observations, with some coordinates corresponding to as many as 15 records. This suggests that a single coordinate point in the dataset may represent a small area rather than a single property.

When constructing neighbourhood features using KNN, if the number of samples corresponding to a particular location exceeds the selected k-value, the distances between these samples will be identical. In such cases, scikit-learn may select only the top k neighbours based on internal sorting during the neighbour selection process, thereby overlooking some samples with identical distances. This situation may lead to a certain degree of instability in the calculation of neighbourhood features and may affect the stability of the model training results.

## Examine the distribution density of data across different regions in the dataset

We analyse the suitability of the KNN method by examining the density of data across different regions within the dataset.

In [ ]:
#Explore the density of data in each region of the dataset
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

# 只用坐标做 KNN
# Use only coordinates for KNN
coords_array = X[['Longitude', 'Latitude']].values

k = 10
knn = NearestNeighbors(n_neighbors=k+1)  # +1 because first neighbor is itself
knn.fit(coords_array)

distances, _ = knn.kneighbors(coords_array)

# 第 k 个邻居的距离（跳过自己）
#The distance of the kth neighbor (excluding oneself)
kth_dist = distances[:, k]

plt.figure(figsize=(8, 6))
scatter = plt.scatter(
    X['Longitude'],
    X['Latitude'],
    c=kth_dist,
    s=10,
    alpha=0.7
)
plt.colorbar(scatter, label=f'Distance to {k}-th nearest neighbour')
plt.title(f'Spatial Variation in Local Sample Density (k = {k})')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.show()

An analysis of the spatial distribution density of the data reveals significant variations in density across different regions. Sample distribution is relatively dense in coastal urban areas (such as Los Angeles and the San Francisco Bay Area), whilst it is comparatively sparse in certain inland regions. As can be seen from the figure, in dense areas, the distance to the kth neighbour is small, whereas in sparse areas, this distance is significantly greater. This implies that when constructing neighbourhood features using a fixed k-value, the actual spatial scope corresponding to different samples may vary considerably.

In areas with a high density of samples, neighbourhood features primarily reflect very localised house price information; whereas in areas with a low density of samples, neighbourhood features may incorporate house price data from much greater distances, thereby reducing the representativeness of the neighbourhood features for local house price structures. This imbalance in spatial density may affect the stability of KNN neighbourhood features and have a certain impact on the model’s predictive performance.

In [1]:
import json
from pathlib import Path
from reportlab.lib.pagesizes import A4
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from reportlab.pdfgen import canvas
from reportlab.lib.units import mm

# ========= 配置 =========
notebook_path = Path("main.ipynb")   # 改成你当前 ipynb 文件名
output_pdf = notebook_path.with_name(notebook_path.stem + "_code_cells.pdf")

# 如果你想支持中文，填入系统里的中文字体路径
# Windows 常见：C:/Windows/Fonts/msyh.ttc
# Mac 常见：/System/Library/Fonts/PingFang.ttc
# Linux 需要换成你机器上的中文字体
font_path = None
font_name = "Helvetica"

if font_path:
    pdfmetrics.registerFont(TTFont("CustomFont", font_path))
    font_name = "CustomFont"

# ========= 读取 notebook =========
with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

code_cells = []
for i, cell in enumerate(nb.get("cells", []), start=1):
    if cell.get("cell_type") == "code":
        source = "".join(cell.get("source", []))
        if source.strip():
            code_cells.append((i, source))

# ========= 写入 PDF =========
page_width, page_height = A4
left_margin = 15 * mm
right_margin = 15 * mm
top_margin = 15 * mm
bottom_margin = 15 * mm
line_height = 6 * mm

c = canvas.Canvas(str(output_pdf), pagesize=A4)
c.setTitle(f"{notebook_path.stem} code cells")

def draw_wrapped_text(c, text, x, y, max_width, font_name, font_size):
    c.setFont(font_name, font_size)
    lines = []
    for paragraph in text.splitlines():
        if paragraph == "":
            lines.append("")
            continue

        current = ""
        for ch in paragraph:
            test = current + ch
            if pdfmetrics.stringWidth(test, font_name, font_size) <= max_width:
                current = test
            else:
                lines.append(current)
                current = ch
        lines.append(current)

    for line in lines:
        if y < bottom_margin:
            c.showPage()
            c.setFont(font_name, font_size)
            y = page_height - top_margin
        c.drawString(x, y, line if line else " ")
        y -= line_height
    return y

y = page_height - top_margin
title_size = 14
text_size = 10
max_text_width = page_width - left_margin - right_margin

for idx, code in code_cells:
    header = f"Code Cell {idx}"
    if y < top_margin + 20 * mm:
        c.showPage()
        y = page_height - top_margin

    c.setFont(font_name, title_size)
    c.drawString(left_margin, y, header)
    y -= 8 * mm

    y = draw_wrapped_text(c, code, left_margin, y, max_text_width, font_name, text_size)
    y -= 8 * mm

c.save()

print(f"PDF 已生成：{output_pdf.resolve()}")

PDF 已生成：/Users/tyb/Desktop/数据科学第二年/MATH2603 Graphs, Networks and Systems/2603-Group-Project/main_code_cells.pdf


In [5]:
import json
from pathlib import Path
from reportlab.lib.pagesizes import A4
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from reportlab.pdfgen import canvas
from reportlab.lib.units import mm

# ========= 配置 =========
notebook_path = Path("main.ipynb")   # 改成你当前 ipynb 文件名
output_pdf = notebook_path.with_name(notebook_path.stem + "_cells.pdf")

# 如果你想支持中文，填入系统里的中文字体路径
# Windows 常见：C:/Windows/Fonts/msyh.ttc
# Mac 常见：/System/Library/Fonts/PingFang.ttc
# Linux 需要换成你机器上的中文字体
font_path = None
font_name = "Helvetica"

if font_path:
    pdfmetrics.registerFont(TTFont("CustomFont", font_path))
    font_name = "CustomFont"

# ========= 读取 notebook =========
with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

cells_to_export = []
for i, cell in enumerate(nb.get("cells", []), start=1):
    cell_type = cell.get("cell_type")
    source = "".join(cell.get("source", []))
    if source.strip() and cell_type in ("code", "markdown"):
        cells_to_export.append((i, cell_type, source))

# ========= 写入 PDF =========
page_width, page_height = A4
left_margin = 15 * mm
right_margin = 15 * mm
top_margin = 15 * mm
bottom_margin = 15 * mm
line_height = 6 * mm

c = canvas.Canvas(str(output_pdf), pagesize=A4)
c.setTitle(f"{notebook_path.stem} cells")

def draw_wrapped_text(c, text, x, y, max_width, font_name, font_size):
    c.setFont(font_name, font_size)
    lines = []
    for paragraph in text.splitlines():
        if paragraph == "":
            lines.append("")
            continue

        current = ""
        for ch in paragraph:
            test = current + ch
            if pdfmetrics.stringWidth(test, font_name, font_size) <= max_width:
                current = test
            else:
                lines.append(current)
                current = ch
        lines.append(current)

    for line in lines:
        if y < bottom_margin:
            c.showPage()
            c.setFont(font_name, font_size)
            y = page_height - top_margin
        c.drawString(x, y, line if line else " ")
        y -= line_height
    return y

y = page_height - top_margin
title_size = 14
text_size = 10
max_text_width = page_width - left_margin - right_margin

for idx, cell_type, content in cells_to_export:
    header = f"{cell_type.capitalize()} Cell {idx}"

    if y < top_margin + 20 * mm:
        c.showPage()
        y = page_height - top_margin

    c.setFont(font_name, title_size)
    c.drawString(left_margin, y, header)
    y -= 8 * mm

    y = draw_wrapped_text(c, content, left_margin, y, max_text_width, font_name, text_size)
    y -= 8 * mm

c.save()

print(f"PDF 已生成：{output_pdf.resolve()}")

PDF 已生成：/Users/tyb/Desktop/数据科学第二年/MATH2603 Graphs, Networks and Systems/2603-Group-Project/main_cells.pdf


本项目的目标是使用 California Housing 数据集构建一个房价的神经网络预测模型。
项目首先建立一个 Baseline 神经网络模型，仅使用原始数据特征进行预测。
随后通过构建 neighbourhood-based features（邻域特征），利用地理邻近关系kNN
提取额外信息，从而构建 Enhanced Model，提升了预测性能。

注：本项目运行时间较长约为15到55分钟，与配置环境和电脑性能相关。去除运行参数择优过程代码可以大幅降低运行时间，可直接运行Part one 第一第二第四和Part three最后一个代码块